[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WesamKhallaf73/pytorch-ground-up-notebooks/blob/main/notebooks/L07.ipynb)

# Lecture 07 — Reductions, Matrix Math & Joining Tensors

*Collapsing a dimension, the two multiplications, and gluing tensors together.*

**PyTorch from Ground Up · Volume I · Chapters 10–11** — companion notebook.

**How to use it:** run the cells from top to bottom. Before each code cell, *predict* what it will print — then run it and compare with the expected output (click ▸ to open it). Every exercise from the lecture is at the end.

By the end you will be able to:
- Reduce a tensor over everything or along one `dim`, and predict the resulting shape.
- Use `keepdim=True` so a reduction still broadcasts against the original.
- Read the two return values of `max`/`min`, and turn scores into predictions with `argmax`.
- Tell `@` (matrix multiply) from `*` (element-wise), and read a simple `einsum`.
- Join with `cat` and `stack`, split with `chunk` and `split`, and choose the right one.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
print('PyTorch', torch.__version__)

## 01 · Reductions and the dim argument

Everything so far kept every element. **Reductions collapse a whole axis** into single numbers. The last major piece of tensor fluency before real models.

<sub>Source: lecture slides 3–8</sub>

**`reductions.py`** — predict the output, then run.

In [ ]:
m = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
print(m.sum())    # 21.0
print(m.mean())   # 3.5
print(m.sum(dim=0))    # tensor([5., 7., 9.])
print(m.sum(dim=1))    # tensor([ 6., 15.])

<details><summary>▸ Expected output</summary>

```
tensor(21.)
tensor(3.5000)
tensor([5., 7., 9.])
tensor([ 6., 15.])
```
</details>

🧪 *Interactive lab in the app:* **Collapse a dimension** — try it there to see this idea move.

## 02 · keepdim: keeping the collapsed axis

By default the reduced axis vanishes. `keepdim=True` **keeps it as a size-1 axis** — so the result still lines up with the original.

<sub>Source: lecture slides 9–12</sub>

**`keepdim.py`** — predict the output, then run.

In [ ]:
row_sum = m.sum(dim=1)
print(row_sum.shape)                   # (2,)
row_sum_k = m.sum(dim=1, keepdim=True)
print(row_sum_k.shape)                 # (2, 1)

# Row-wise normalization
normed = m / m.sum(dim=1, keepdim=True)
print(normed)
print(normed.sum(dim=1))   # each row sums to 1

<details><summary>▸ Expected output</summary>

```
torch.Size([2])
torch.Size([2, 1])
tensor([[0.1667, 0.3333, 0.5000],
        [0.2667, 0.3333, 0.4000]])
tensor([1., 1.])
```
</details>

> ⚠️ **Watch out — Reduce, then reuse → keepdim**  
> Whenever you reduce a tensor and then immediately use the result in an arithmetic operation with the original, you almost certainly need `keepdim=True`.
>
> If you forget, the shapes may still broadcast — but to something larger than you intended.

## 03 · max, min and argmax

With a `dim`, `max` and `min` hand back **two tensors**: the values, and where they were.

<sub>Source: lecture slides 13–14</sub>

**`argmax.py`** — predict the output, then run.

In [ ]:
vals, idxs = m.max(dim=1)
print(vals)   # tensor([3., 6.])
print(idxs)   # tensor([2, 2])
print(m.argmax(dim=1))   # tensor([2, 2])
print(m.argmax())        # 5 — flat index

<details><summary>▸ Expected output</summary>

```
tensor([3., 6.])
tensor([2, 2])
tensor([2, 2])
tensor(5)
```
</details>

## 04 · Element-wise math, matmul and einsum

Most PyTorch math works **element by element** and keeps the shape. Arithmetic, powers, exponentials, clamps, comparisons — same names, same behaviour and same broadcasting rules as NumPy.

<sub>Source: lecture slides 15–22</sub>

**`elementwise.py`** — predict the output, then run.

In [ ]:
a = torch.tensor([1.0, 4.0, 9.0])
print(a + 1)                 # [2, 5, 10]
print(a ** 0.5)              # [1, 2, 3]  (sqrt)
print(torch.clamp(a, 2, 5))  # [2, 4, 5]
print(a > 3)                 # [False, True, True]

<details><summary>▸ Expected output</summary>

```
tensor([ 2.,  5., 10.])
tensor([1., 2., 3.])
tensor([2., 4., 5.])
tensor([False,  True,  True])
```
</details>

🧪 *Interactive lab in the app:* **@ versus *** — try it there to see this idea move.

> ⚠️ **Watch out — * is not @**  
> `*` is element-wise multiplication. `@` is matrix multiplication. Confusing them is a rite-of-passage bug.
>
> If your shapes explode unexpectedly, check which operator you used.

_Stand-in: the next listing uses names from your own project (a model, a data loader…). This cell defines small ones so it runs here._

In [ ]:
A = torch.randn(2, 3)
B = torch.randn(3, 4)


**`einsum_intro.py`** — predict the output, then run.

In [ ]:
# Matrix multiply via einsum
C = torch.einsum("ik,kj->ij", A, B)
# Batch trace (sum of diagonals)
t = torch.randn(5, 3, 3)
traces = torch.einsum("bii->b", t)
print(traces.shape)   # (5,)

<details><summary>▸ Expected output</summary>

```
torch.Size([5])
```
</details>

## 05 · Joining: cat and stack

Real code constantly **combines tensors** — and splits them apart again. A batch from single images; features from different sources. Four operations cover every case.

<sub>Source: lecture slides 24–33</sub>

🧪 *Interactive lab in the app:* **cat or stack?** — try it there to see this idea move.

**`build_batch.py`** — predict the output, then run.

In [ ]:
# Three "images" of shape (3, 28, 28)
imgs = [torch.randn(3, 28, 28) for _ in range(3)]
batch = torch.stack(imgs, dim=0)
print(batch.shape)   # (3, 3, 28, 28)

<details><summary>▸ Expected output</summary>

```
torch.Size([3, 3, 28, 28])
```
</details>

> ⚠️ **Watch out — Everything except the cat axis must match**  
> All tensors in the list must have the same number of dimensions, and every dimension except the cat dimension must match in size.
>
> A (2, 3) and a (2, 4) can be cat’d along dim 1 — giving (2, 7) — but not along dim 0.

## 06 · Splitting: chunk and split

`chunk` asks **how many pieces?** `split` asks **what sizes?** Both return views — no data is copied.

<sub>Source: lecture slides 34–39</sub>

**`splitting.py`** — predict the output, then run.

In [ ]:
t = torch.arange(12).reshape(4, 3)
parts = torch.chunk(t, chunks=2, dim=0)
for p in parts:
    print(p.shape)

t = torch.arange(10)
# Split into pieces of size 3, 3, and 4
for p in torch.split(t, [3, 3, 4]):
    print(p)
# Split into pieces of size 3 each
print([p.shape for p in torch.split(t, 3)])

<details><summary>▸ Expected output</summary>

```
torch.Size([2, 3])
torch.Size([2, 3])
tensor([0, 1, 2])
tensor([3, 4, 5])
tensor([6, 7, 8, 9])
[torch.Size([3]), torch.Size([3]), torch.Size([3]), torch.Size([1])]
```
</details>

> ⚠️ **Watch out — Sizes must add up**  
> When you pass a list to `split`, the sizes must sum to the dimension’s length. When you pass a single integer, the last piece may be smaller — just like `chunk`.

## The rule — worth remembering

**Part One — reductions & math**

1. A reduction collapses a dimension. The dim you name is the one that disappears.
2. Use `keepdim=True` when the result must broadcast against the original tensor.
3. `max(dim)` and `min(dim)` return two tensors: values and indices.
4. `argmax` returns only the indices — how you turn scores into predictions.
5. `@` is matrix multiplication: (M, K) @ (K, N) → (M, N). `*` is element-wise.
6. `einsum` is a compact notation for reductions, matmuls and more — learn it when you reach attention.

**Part Two — joining & splitting**

1. `cat(tensors, dim)` joins along an existing axis. The cat dimension grows; all others must match.
2. `stack(tensors, dim)` joins along a new axis. All inputs must have the same shape.
3. Stack is how you build a batch from individual samples: `stack(samples, dim=0)`.
4. `chunk(t, n, dim)` splits into n equal pieces. `split(t, sizes, dim)` splits by exact sizes.
5. Both `chunk` and `split` return views — no data is copied.
6. Cat undoes split/chunk. Stack undoes unbind.

## Pause and try

Work each exercise in the empty cell below it. Open the worked answer only after you have tried.

### Exercise 01 · Reductions

Normalize each column of a (5, 3) matrix to zero mean — what shape should the mean have? Use `keepdim=True`. Then take `torch.randn(8, 10)` (8 samples, 10 classes) and use `argmax(dim=1)` to get predicted labels — what is the shape of the result?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

`m - m.mean(dim=0, keepdim=True)`; the mean has shape (1, 3) so it stretches down the 5 rows. `argmax(dim=1)` on (8, 10) gives shape (8,) — one label per sample.
</details>

### Exercise 02 · Math

Multiply a (2, 3) matrix by a (3, 5) matrix using `@` — what is the output shape? Now try `*` instead: what happens, and why? Then rewrite `m.sum(dim=0)` using `torch.einsum` and verify with `torch.allclose()`.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

`@` gives (2, 5): the inner 3s match and disappear. `*` raises a RuntimeError, because element-wise multiplication broadcasts and (2, 3) vs (3, 5) can’t be aligned. `torch.einsum("ij->j", m)` sums away the row axis.
</details>

### Exercise 03 · Joining & splitting

Build a batch of five “images” of shape (3, 16, 16) using `stack`, confirm it is (5, 3, 16, 16), then split it back into five individual images with `chunk` — each (1, 3, 16, 16). How would you remove the leading 1? Look up `torch.unbind`: how does it differ from `chunk`?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

`torch.stack(imgs, dim=0)` → (5, 3, 16, 16); `torch.chunk(batch, 5, dim=0)` gives five (1, 3, 16, 16) pieces; `.squeeze(0)` removes the leading 1. `torch.unbind(batch, dim=0)` removes the dimension itself, returning five (3, 16, 16) tensors directly — it is the true inverse of `stack`.
</details>

### Exercise 04 · Take it apart, put it back

Create a (6, 4) tensor. Split it into three equal pieces along dim=0 using `chunk`. Recombine them with `cat` along dim=0. Verify the result equals the original with `torch.equal`.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

`parts = torch.chunk(t, 3, dim=0)` gives three (2, 4) tensors; `torch.equal(torch.cat(parts, dim=0), t)` is True. Cat undoes chunk.
</details>

### Interview-style

“Your model outputs a tensor of shape (32, 10) — 32 samples, 10 class scores. How do you get the predicted class for each sample, and what is the resulting shape?”

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

Call `scores.argmax(dim=1)`, which collapses the class dimension and returns the index of the highest score in each row. The result has shape (32,) — one integer label per sample.
</details>

### Interview-style

“What is the difference between `torch.cat` and `torch.stack`?” — two sentences, mentioning dimensions.

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

`cat` joins tensors along a dimension that already exists, so that dimension grows and the number of dimensions stays the same. `stack` joins tensors of identical shape along a brand-new dimension, so the result has one more dimension than its inputs.
</details>